<a href="https://colab.research.google.com/github/bzhuang2-create/SURF---MC-Simulation-for-Educational-Research/blob/main/Main_Function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Main Function
'''
The main function which implmentes the Monte Carlo simulation according to the Metropolis algorithm
Volume, number of atoms, and temperature are fixed. Periodic boundary conditions are used.

density: number density of the atoms (atoms per A^3)
no_atoms: total number of atoms
species: list of tuples of the type (Atomic Symbol, Mol Fraction)
iterations: the mnaximum number of iterations the  simulation will complete
temperature: temperature (K)
energy: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs
n_warmup: after this many iterations, the simulation is assumed to be at equilibrium
'''
def MC_main(density, no_atoms, species, iterations, temp,
            energy = lennard_jones, params = LJ_avg_array, energy_derivative = lennard_jones_derivative, tail_correction_fxn = frenkel_smit_tail_correction,
            n_warmup = 1000000, display_energy_profile = False, cube_start = False, anneal = False, reference_start = False):

  MC = MC_initial(no_atoms, density, species, cube_start, reference_start)

  energy_fxn = None
  if(energy is None):
    energy_fxn = lennard_jones
  else:
    energy_fxn = energy

  length = density_to_length(no_atoms, density)
  r_cutoff = length / 2.01
  default_step_size = 0.125
  step_size = default_step_size

  atom_names = MC.get_chemical_symbols()
  atom_numbers = MC.get_atomic_numbers()

  frames = []
  energy_array = []
  current_position = MC.get_positions()

  # very high temperature to remove system from local minimum
  # the simulation will start annealing at this temperature whenever annealing begins
  anneal_temp = temp + 300
  anneal_temp_step_size = 0.90 # whenever a step of annealing is finished, decrease the temperature by this factor (exponential cooling)
  given_temp = temp # annealing will end once we reach this temperature
  anneal_iterations = 2.5e4 # number of iterations per stage of annealing
  anneal_step_size = 1.00 * step_size # testing if larger step sizes help annealing

  currently_annealing = anneal # whether the simulation is currently annealing
  current_anneal_iteration = 0
  current_anneal_temperature = anneal_temp

  if(anneal):
    temperature = anneal_temp
    current_anneal_iterations = anneal_iterations
    step_size = anneal_step_size
  else:
    temperature = temp

  # this logic determines how the final iterations to find RDF averages and pressure averages are spaced
  final_samples = 25 # the number of final samples to take for the RDFs
  final_sparse = 2000 # how many interations to go through between each final sample
  final_iterations = final_samples * final_sparse
  no_bins = int(12.5 * length)


  total_potential_energy = total_energy(atom_numbers, current_position, no_atoms, length, energy_fxn, params)

  # only used when the user wants to record energy profiles
  energy_profile_sparse = 80
  energy_profile_cutoff = 500
  energy_profile_array = np.zeros(int(energy_profile_cutoff + (iterations - energy_profile_cutoff) / energy_profile_sparse), dtype = object)
  energy_profile_bin_size = 7.5e-2

  positive_bins = 5
  negative_bins = 20
  energy_profile_bin_number = positive_bins + negative_bins
  energy_profile_index = 0

  move_accepted_array = np.zeros(iterations, dtype = bool)

  # iterates over ther user-specified number of iterations
  for i in tqdm(range(iterations), leave = False, desc = "Main Iterations: "):
    if(not currently_annealing):
      assert(step_size == default_step_size)
      assert(temperature == temp)

    current_position, total_potential_energy, energy_array, move_accepted_array = iterate(no_atoms, atom_numbers, current_position, length, energy_fxn, params, step_size,
                                                                                          n_warmup, energy_array, temperature, i, total_potential_energy, move_accepted_array, True)

    # writing to OVITO - may need to change later to make it compatible with live display
    if i % 5000 == 0:
      MC.set_positions(current_position)
      frame = MC.copy()
      frame.info["step"] = i
      frame.info["temperature"] = temperature
      frames.append(frame)

    if(display_energy_profile):
      if(i < energy_profile_cutoff or i % energy_profile_sparse == 0):
        energy_profile_array[energy_profile_index] = find_energy_distribution(energy_profile_bin_size, energy_profile_bin_number, negative_bins, no_atoms, current_position, atom_numbers, length, energy_fxn, params)
        energy_profile_index += 1

    if(currently_annealing):
      current_anneal_iteration -= 1

      if(current_anneal_iteration < 0):

        current_anneal_iteration = anneal_iterations
        current_anneal_temperature *= anneal_temp_step_size

        if(current_anneal_temperature < given_temp):
          currently_annealing = False
          temperature = given_temp
          step_size = default_step_size
        else:
          temperature = current_anneal_temperature


  unique_species = (pd.unique(pd.Series(atom_names)))
  unique_numbers = (pd.unique(pd.Series(atom_numbers)))
  len_species = len(unique_species)

  R_bins = bins_to_distance(length, no_bins)
  raw_RDFs = np.empty((final_samples, cantor_pair(len_species, len_species)), dtype = object)

  virial_pressure_array = []

  for final in tqdm(range(final_iterations), leave = False, desc = "Finding RDFs: "):
    current_position, total_potential_energy, energy_array, move_accepted_array = iterate(no_atoms, atom_numbers, current_position, length, energy_fxn, params, step_size,
                                                                                          n_warmup, energy_array, temperature, final, total_potential_energy, move_accepted_array, False)

    # only take a smple RDF every final_sparse iterations
    if(final % final_sparse == 0):
      index = final // final_sparse
      for i in range(len_species):
        for j in range(len_species):
          species1 = unique_species[i]
          species2 = unique_species[j]

          RDF = partial_RDF(species1, species2, atom_names, no_atoms, current_position, length, R_bins)
          raw_RDFs[index][cantor_pair(i, j)] = RDF

      virial_pressure_term = virial_pressure(atom_numbers, current_position, length, energy_fxn, params)
      virial_pressure_total = density * kB * temperature + virial_pressure_term
      virial_pressure_array.append(virial_pressure_total)

  averaged_species_RDFs = np.zeros(cantor_pair(len_species, len_species), dtype = object)

  for i in range(len_species):
    for j in range(len_species):
      sum = np.zeros(no_bins)
      for k in range(final_samples):
        sum += raw_RDFs[k][cantor_pair(i, j)]
      averaged_species_RDFs[cantor_pair(i, j)] = sum / final_samples


  manual_RDF = np.zeros(len(averaged_species_RDFs[cantor_pair(0, 0)]))

  for i in range(len_species):
    for j in range(len_species):
      fraction = find_fractions(species, unique_species[i]) * find_fractions(species, unique_species[j])
      manual_RDF += fraction * averaged_species_RDFs[cantor_pair(i, j)]
  # this is the total RDF, not the individual species RDF


  RDF_pressure_term = 0

  for i in range(len_species):
    for j in range(i, len_species):

      number1 = unique_numbers[i]
      number2 = unique_numbers[j]

      fraction1 = find_fractions(species, unique_species[i])
      fraction2 = find_fractions(species, unique_species[j])

      integral_term = fraction1 * fraction2 * density * density * pressure_equation(number1, number2, averaged_species_RDFs[cantor_pair(i, j)], R_bins, params)
      tail_term = tail_correction(tail_correction_fxn, number1, number2, fraction1, fraction2, params, r_cutoff, density)
      RDF_pressure_term += integral_term + tail_term


  RDF_pressure_total = density * kB * temperature - 2 / 3 * np.pi * RDF_pressure_term
  virial_pressure_average = np.average(virial_pressure_array)


  #ASE RDF (not species specific)
  RDF, R_bins = get_rdf(MC, length / 2.01, nbins = no_bins)

  # consider changing or removing this for updated OVITO display
  #write(f"P = {virial_pressure_average} N = {no_atoms} rho = {density}.extxyz", frames)
  write(f"output.extxyz", frames)

  if(display_energy_profile):

    return (MC, RDF, manual_RDF, averaged_species_RDFs, R_bins, no_bins, energy_profile_array, energy_array, (RDF_pressure_total, virial_pressure_average), move_accepted_array)
    # energy_profile_array: histogram, bin_size, no_bins, cutoff_bin

  # note: MC contains the positions of the atoms
  return (MC, RDF, manual_RDF, averaged_species_RDFs, R_bins, no_bins, energy_array, (RDF_pressure_total, virial_pressure_average), move_accepted_array)
